In [ ]:
from mxdataspark import MXData
from mxdataspark.core import MXDataClient


In [ ]:
from importlib.metadata import version

print(f'MXData Spark package version: {version("mxdataspark")}')


In [ ]:
# DBTITLE 1,Widget Parameters
dbutils.widgets.text("load_group", "-1")
dbutils.widgets.text("source_id", "-1")
dbutils.widgets.text("flow_id", "-1")
dbutils.widgets.text("job_concurrency", "")
dbutils.widgets.text("run_id", "-1")


In [ ]:
load_group = int(dbutils.widgets.get("load_group"))
source_id = int(dbutils.widgets.get("source_id"))
flow_id = dbutils.widgets.get("flow_id")
job_concurrency = dbutils.widgets.get("job_concurrency")
run_id = dbutils.widgets.get("run_id")
job_concurrency = int(job_concurrency) if int(job_concurrency or 0) > 0 else None


In [ ]:
try:
    client = MXDataClient()
    extract_configs = client.get_extraction_object_list(load_group=load_group, source_id=source_id, flow_id=flow_id)

    notebook_output = {
        "errorInfo": None  # No error occurred
    }
except Exception as e:
    # Full error message
    full_error_message = str(e)

    # Define the error messages to look for
    error_messages = [
        'Duplicate object names across multiple schema for this source have been detected! To avoid data processing failures, please update objectlist.tablename to ensure these objects are uniquely named.',
        'Flow is not of type: ACQUISITION',
        'One or more failed extractions are in the history for this object.  Since ordering is important, you must investigate and clear them to resume extraction.'
    ]

    # Initialize a variable to hold the extracted relevant error message
    relevant_error = None

    # Check if any of the relevant error messages are in the full error message
    for message in error_messages:
        if message in full_error_message:
            relevant_error = message
            break  # Exit loop once the relevant error is found

    # If no relevant error message was found, use a default message
    if relevant_error is None:
        relevant_error = 'An unknown error occurred. Kindly check the Notebook for more details'
        print(full_error_message)

    # Prepare notebook_output with the relevant error message
    notebook_output = {
        "errorInfo": relevant_error
    }

    # Raise the exception with the relevant error message to fail the ADF activity
    raise Exception(notebook_output['errorInfo'])


In [ ]:
print("MXData.queue_extraction_batch Configs:")
print(f"{extract_configs=}")
print(f"{job_concurrency=}")


In [ ]:
# DBTITLE 1,Main
metrics = (
    MXData.source("batch")
    .queue_extraction_batch(extract_configs)
    .process_batch(workers=job_concurrency, adf_runid=run_id)
    .metrics("dict")
)


In [ ]:
FAIL = False

for metric in metrics:
    if not metric["extractionSuccess"]:
        FAIL = True


In [ ]:
if FAIL:
    print(f"An extraction has failed! Check the Metrics.")
    for metric in metrics:
        if not metric["extractionSuccess"]:
            print(metric)
    raise ValueError("An extraction has failed! Check ObjectExtract. Exiting... ")
